# Data analysis

In this notebook, we will analyze our dataset for typical statistics parameters. To investigate dependencies and assess the distribution of the data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../../data/diabetes.csv")
df.head()

In [ ]:
df.info()
df.describe().drop(index=["count"], columns=["Outcome"])

Several parameters contain biologically impossible values (e.g., Blood Pressure equal to 0.0). This indicates that missing measurements were recorded as 0s in this dataset. 

Feeding false zeros to a machine learning model will negatively impact its performance. To ensure our binary classification model trains correctly, we must handle these missing values using the KNN imputation algorithm before training.

## Applying KNN Imputation
Since we already have a trained KNN imputer and a preprocessing script in our `database/` module, we can import them here to clean our dataset. This will make all our subsequent histograms, boxplots, and heatmaps much more accurate!

In [ ]:
import sys
from pathlib import Path

# Add the root project directory to Python path so we can import 'database'
sys.path.append(str(Path.cwd().parent.parent))

from database.preprocessing import fill_missing_values, load_imputer

# Load the pre-trained imputer from the database folder
imputer = load_imputer("../../database/imputer.joblib")

# Clean the dataframe
df = fill_missing_values(df, imputer)

# Check the statistics again
df.describe().drop(index=["count"], columns=["Outcome"])

In [ ]:
features = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes_flat = axes.flatten()

for i, feature in enumerate(features):
    sns.histplot(
        data=df, x=feature, kde=True, color="#10b981", ax=axes_flat[i], bins=30
    )

    axes_flat[i].set_title(f"Distribution: {feature}")
    axes_flat[i].set_xlabel("Value")
    axes_flat[i].set_ylabel("Number of patients")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature correlation matrix")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i, feature in enumerate(features):
    sns.boxplot(
        data=df,
        x="Outcome",
        y=feature,
        ax=axes.flatten()[i],
        palette=["#10b981", "#f43f5e"],
    )
    axes.flatten()[i].set_title(f"{feature} vs Outcome")
plt.tight_layout()
plt.show()

In [ ]:
old_group = df[df["Age"] > 50].groupby("Outcome").size()
young_group = df[df["Age"] <= 50].groupby("Outcome").size()

fig, axes = plt.subplots(1, 2, figsize=(12, 6))  # index, column, figure
colors_ = ["#10b981", "#f43f5e"]
labels_ = ["Healthy (0)", "Diabetic (1)"]

old_group.plot.pie(
    ax=axes[0],
    autopct="%1.1f%%",
    labels=labels_,
    colors=colors_,
    startangle=90,
    ylabel="",
)
axes[0].set_title("Patients > 50 old")

young_group.plot.pie(
    ax=axes[1],
    autopct="%1.1f%%",
    labels=labels_,
    colors=colors_,
    startangle=90,
    ylabel="",
)
axes[1].set_title("Patients <= 50 old")

plt.tight_layout()
plt.show()

## EDA Conclusions

1. **Data Quality & Missing Values:** The dataset initially contained impossible zero values for biological measurements (e.g., blood pressure, BMI, glucose). Using the KNN Imputer algorithm allowed us to reliably handle these missing values and prepare the data for modeling.
2. **Feature Distributions:** After imputation, the features display much more realistic distributions. For instance, insulin levels are heavily right-skewed, while features like blood pressure approximate a normal distribution more closely.
3. **Impact of Age:** The pie charts clearly demonstrate that age is a significant factor in diabetes prevalence. The group of patients older than 50 has a noticeably higher percentage of diabetic cases compared to the younger group (<= 50 years old).
